# EchoFind - Embedding Visualization

This notebook visualizes the learned audio embeddings using t-SNE and UMAP, colored by genre labels.

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import config
from evaluate import linear_probe_evaluation

# Check UMAP availability
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("Warning: umap-learn not installed. UMAP visualization will be skipped.")

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Step 1: Extract Embeddings

Run linear probe evaluation to extract embeddings from the trained encoder.

In [ ]:
# Run evaluation to get embeddings
results = linear_probe_evaluation()

if results:
    embeddings = results['embeddings']
    labels = results['labels']
    track_ids = results['track_ids']
    f1_score = results['f1_score']
    
    print(f"Extracted {len(embeddings)} embeddings")
    print(f"Embedding dimension: {embeddings.shape[1]}")
    print(f"Number of classes: {len(np.unique(labels))}")
    print(f"Linear probe F1-score: {f1_score:.4f}")
else:
    print("Could not extract embeddings. Make sure encoder is trained and labels are available.")

## Step 2: t-SNE Visualization

In [ ]:
# Subsample if too many points (for faster visualization)
if len(embeddings) > 5000:
    print(f"Subsampling from {len(embeddings)} to 5000 points...")
    indices = np.random.choice(len(embeddings), 5000, replace=False)
    embeddings_subset = embeddings[indices]
    labels_subset = labels[indices]
else:
    embeddings_subset = embeddings
    labels_subset = labels

# Apply t-SNE
print("Computing t-SNE...")
tsne = TSNE(n_components=2, random_state=config.RANDOM_SEED, perplexity=30, n_iter=1000)
embeddings_2d_tsne = tsne.fit_transform(embeddings_subset)

# Plot
plt.figure(figsize=(12, 10))
unique_labels = np.unique(labels_subset)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))

for i, label in enumerate(unique_labels):
    mask = labels_subset == label
    plt.scatter(
        embeddings_2d_tsne[mask, 0],
        embeddings_2d_tsne[mask, 1],
        c=[colors[i]],
        label=f'Class {label}',
        alpha=0.6,
        s=20
    )

plt.title('Embedding Visualization (t-SNE)', fontsize=16)
plt.xlabel('Dimension 1', fontsize=12)
plt.ylabel('Dimension 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../results/embeddings_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 3: UMAP Visualization

In [ ]:
# Apply UMAP
if UMAP_AVAILABLE:
    print("Computing UMAP...")
    umap_reducer = umap.UMAP(n_components=2, random_state=config.RANDOM_SEED, n_neighbors=15, min_dist=0.1)
    embeddings_2d_umap = umap_reducer.fit_transform(embeddings_subset)
    
    # Plot
    plt.figure(figsize=(12, 10))
    for i, label in enumerate(unique_labels):
        mask = labels_subset == label
        plt.scatter(
            embeddings_2d_umap[mask, 0],
            embeddings_2d_umap[mask, 1],
            c=[colors[i]],
            label=f'Class {label}',
            alpha=0.6,
            s=20
        )
    
    plt.title('Embedding Visualization (UMAP)', fontsize=16)
    plt.xlabel('Dimension 1', fontsize=12)
    plt.ylabel('Dimension 2', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig('../results/embeddings_umap.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("UMAP not available. Install with: pip install umap-learn")